# Grammy 2027 forecast — nomination training dataset

This notebook builds the table used to estimate **nomination probabilities** (Stage 1): Billboard chart features, previous Grammy record,
and nominations at three other music awards (American Music Awards, MTV Video Music Awards, iHeartRadio Music Awards).

* **One row = artist × season × category.** The pool of candidates in a season is every artist who appeared on the
  Billboard Hot 100 or Billboard 200 during that season's eligibility window.
* **Target `nominated`:** 1 if the artist is credited on a nomination in that category at the ceremony of that season, 0 otherwise.
* **Season `t`:** the eligibility year ending in `t`; nominations are announced in November of `t`, the ceremony is held in
  February of `t+1`. Example: season 2024 → 67th Grammys (February 2025). Season 2026 → 69th Grammys (February 2027), the forecast target.
* **Eligibility window:** approximated as 1 October of `t−1` to 30 September of `t` (the official windows differ by a few weeks between years).
* **Features** use only charts inside the window and Grammy results of earlier seasons, so every feature is known before the nominations are announced.

The eight categories include their pre-2012 predecessors (see the mapping below), so the history starts in season 2000.

## 1. Data sources

| File | Source | Use |
|---|---|---|
| `hot-100-current.csv`, `billboard-200-current.csv` | [utdata/rwd-billboard-data](https://github.com/utdata/rwd-billboard-data), commit `f5801b0`, downloaded below | weekly chart positions |
| `grammy_winners_2.csv` | Grammy nominees and winners by category, 1958–2024 seasons (scraped from [grammy.com](https://www.grammy.com/awards)) | nominations and winners for seasons 2000–2024 |
| `data/grammy_nominees_wikipedia.csv` | performer of each nominated work, transcribed from the Wikipedia pages of each category ([example](https://en.wikipedia.org/wiki/Grammy_Award_for_Best_Pop_Solo_Performance)) | the Grammy file gives the performer only for winners in several categories |
| `ama_pop_features_filtered_1974_2026.csv`, `vma_pop_features_1984_2026.csv`, `iheart_pop_features_2014_2026.csv` | nominees and winners of the American Music Awards, MTV VMAs and iHeartRadio Music Awards in pop-relevant categories, one row per artist and ceremony year | other-award features (section 9) |
| `data/grammy_68th_2026.csv` | 68th Grammys nominees and winners ([list of winners](https://www.myjoyonline.com/winners-and-nominees-at-the-2026-grammy-awards/)) | season 2025, missing from the Grammy file |

Every file is checked against its SHA-256 hash so that the notebook always runs on the same data.

In [1]:
import hashlib, re, unicodedata, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd

DATA = Path("data"); DATA.mkdir(exist_ok=True)
BB_COMMIT = "f5801b03d5e629845e65213ff236df98f1aae3fd"
BB_URL = f"https://raw.githubusercontent.com/utdata/rwd-billboard-data/{BB_COMMIT}/data-out/"

SOURCES = {
    "hot-100-current.csv":      ("3f2a274dd7cb26217c3bce8fb2e761ed3e131c5da6c00744a39c070e8b47dc20", BB_URL + "hot-100-current.csv"),
    "billboard-200-current.csv": ("a22e2ef47e3bb8e27111bca7f824f2dcd06d1627bbf508bcbcfe147803154380", BB_URL + "billboard-200-current.csv"),
    "grammy_winners_2.csv":      ("de6e9ba1a6371d6c2e110a43e364628ef1e14647c2d673f21a2d73db9a3f123d", None),
    "grammy_nominees_wikipedia.csv":          (None, None),
    "grammy_68th_2026.csv":                   (None, None),
    "ama_pop_features_filtered_1974_2026.csv": ("f66e1df563aabd8a576f10650e905172a4ac3ff1a1c4680c94d83e307fcd6cc5", None),
    "vma_pop_features_1984_2026.csv":          ("0860a2e860440b42c31fd76a8f27920cfd9626558f610ef8c2c01a7431a915a6", None),
    "iheart_pop_features_2014_2026.csv":       ("0f751aa9270476805dfa4d2684331164a5b628958a13179a4387ad0661f7580e", None),
}

def fetch(name):
    sha, url = SOURCES[name]
    path = DATA / name
    if not path.exists():
        assert url is not None, f"put {name} into {DATA}/"
        urllib.request.urlretrieve(url, path)
    assert path.exists(), f"put {name} into {DATA}/"
    if sha is not None:
        assert hashlib.sha256(path.read_bytes()).hexdigest() == sha, f"hash mismatch: {name}"
    return path

hot  = pd.read_csv(fetch("hot-100-current.csv"),      parse_dates=["chart_week"]).assign(chart="hot100")
b200 = pd.read_csv(fetch("billboard-200-current.csv"), parse_dates=["chart_week"]).assign(chart="b200")
grammy = pd.read_csv(fetch("grammy_winners_2.csv"))
wiki = pd.read_csv(fetch("grammy_nominees_wikipedia.csv"), sep="|")
g68  = pd.read_csv(fetch("grammy_68th_2026.csv"), sep="|")
awards_raw = {a: pd.read_csv(fetch(f), encoding="utf-8-sig") for a, f in [
    ("ama", "ama_pop_features_filtered_1974_2026.csv"),
    ("vma", "vma_pop_features_1984_2026.csv"),
    ("iheart", "iheart_pop_features_2014_2026.csv")]}
print(f"Hot 100: {hot.chart_week.min().date()} – {hot.chart_week.max().date()}, {len(hot):,} rows")
print(f"Billboard 200: {b200.chart_week.min().date()} – {b200.chart_week.max().date()}, {len(b200):,} rows")
print(f"Grammy file: seasons {grammy.year.min()}–{grammy.year.max()}; Wikipedia performers: {len(wiki)}; 68th Grammys: {len(g68)}")
for a, d in awards_raw.items():
    print(f"{a.upper()}: ceremonies {d.year.min()}–{d.year.max()}, {len(d)} artist-years")

Hot 100: 1958-08-04 – 2026-10-17, 355,900 rows
Billboard 200: 1967-04-15 – 2026-10-17, 620,891 rows
Grammy file: seasons 1958–2024; Wikipedia performers: 1048; 68th Grammys: 52
AMA: ceremonies 1974–2026, 800 artist-years
VMA: ceremonies 1984–2026, 1089 artist-years
IHEART: ceremonies 2014–2026, 455 artist-years


## 2. Settings and category mapping

In 2012 the Recording Academy merged several pop categories. We map each predecessor to its current category:
Best Pop Solo Performance ← Female and Male Pop Vocal Performance; Best Pop Duo/Group Performance ← Pop Performance by a Duo or Group
with Vocals and Pop Collaboration with Vocals; Best Music Video ← Best Short Form Music Video.
Before 2012 the solo category had two winners per season (female and male); for nominations this only changes the number of places.

In [2]:
FIRST_SEASON, LAST_LABELLED, FORECAST_SEASON = 2000, 2025, 2026

CATEGORY_MAP = {
    "album_of_the_year":  ["Album Of The Year", "Album Of the Year"],
    "record_of_the_year": ["Record Of The Year"],
    "song_of_the_year":   ["Song Of The Year"],
    "best_new_artist":    ["Best New Artist"],
    "pop_vocal_album":    ["Best Pop Vocal Album", "Pop Vocal Album"],
    "pop_solo":           ["Best Pop Solo Performance", "Pop Solo Performance",
                           "Best Female Pop Vocal Performance", "Best Male Pop Vocal Performance",
                           "Female Pop Vocal Performance", "Male Pop Vocal Performance"],
    "pop_duo_group":      ["Best Pop Duo/Group Performance", "Pop Duo/Group Performance",
                           "Best Pop Performance By A Duo Or Group With Vocal",
                           "Best Pop Performance By A Duo Or Group With Vocals",
                           "Pop Performance By A Duo Or Group With Vocals",
                           "Best Pop Collaboration With Vocals", "Pop Collaboration With Vocals"],
    "music_video":        ["Best Music Video", "Best Short Form Music Video", "Music Video/Short Form"],
}
RAW_TO_CAT = {raw: cat for cat, raws in CATEGORY_MAP.items() for raw in raws}
CATS = list(CATEGORY_MAP)
ALBUM_CATS = {"album_of_the_year", "pop_vocal_album"}

def window(t):
    # eligibility window of season t (approximation)
    return pd.Timestamp(f"{t - 1}-10-01"), pd.Timestamp(f"{t}-09-30")

## 3. Artist names

Names are compared through a normalized key (ASCII, lower case, letters and digits only), so that "Beyoncé" and "Beyonce" match.
Billboard credits such as "Taylor Swift Featuring Post Malone" are split into individual artists: every credited artist gets the song.
"A Featuring B" and "A With B" are always split. "A & B", "A, B", "A x B" and "A And B" are split only if at least one piece charts
under its own name **and** every other piece looks like a separate artist (it charts on its own, or it is a name of two or more words
that does not start with "The"). This keeps groups such as "Mumford & Sons", "Florence And The Machine" or "Tank And The Bangas" whole
while splitting duets such as "Cynthia Erivo & Ariana Grande".

In [3]:
def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    s = re.sub(r"[\[\(].*?[\]\)]", "", s)       # drop bracketed parts, e.g. (Live), [From "Barbie"]
    s = re.sub(r"^the\s+", "", s.strip())
    return re.sub(r"[^a-z0-9]", "", s)

MAIN_SPLIT = re.compile(r"\s+(?:featuring|feat\.?|ft\.?|duet with|with)\s+", re.I)
JOIN_SPLIT = re.compile(r"\s*,\s*|\s+&\s+|\s+x\s+|\s+and\s+|\s+\+\s+", re.I)
standalone = {norm(p) for p in pd.concat([hot.performer, b200.performer]).unique()
              if not MAIN_SPLIT.search(p) and not JOIN_SPLIT.search(p)}

def looks_like_artist(piece):
    p = piece.strip()
    return norm(p) in standalone or (len(p.split()) >= 2 and not p.lower().startswith("the "))

def split_credit(performer):
    names = []
    for part in MAIN_SPLIT.split(str(performer)):
        pieces = [x for x in JOIN_SPLIT.split(part) if x.strip()]
        n_alone = sum(norm(x) in standalone for x in pieces)
        ok = len(pieces) > 1 and (n_alone >= 2 or (n_alone == 1 and all(looks_like_artist(x) for x in pieces)))
        names += pieces if ok else [part]
    return [n.strip() for n in names if norm(n)]

for example in ["Taylor Swift Featuring Post Malone", "Lady Gaga & Bruno Mars", "Cynthia Erivo & Ariana Grande",
                "Zedd, Maren Morris & Grey", "Mumford & Sons", "Florence And The Machine", "Tank And The Bangas", "Earth, Wind & Fire"]:
    print(f"{example!r:40} -> {split_credit(example)}")

'Taylor Swift Featuring Post Malone'     -> ['Taylor Swift', 'Post Malone']
'Lady Gaga & Bruno Mars'                 -> ['Lady Gaga', 'Bruno Mars']
'Cynthia Erivo & Ariana Grande'          -> ['Cynthia Erivo', 'Ariana Grande']
'Zedd, Maren Morris & Grey'              -> ['Zedd', 'Maren Morris', 'Grey']
'Mumford & Sons'                         -> ['Mumford & Sons']
'Florence And The Machine'               -> ['Florence And The Machine']
'Tank And The Bangas'                    -> ['Tank And The Bangas']
'Earth, Wind & Fire'                     -> ['Earth, Wind & Fire']


## 4. Chart features per artist and season

For every artist and season we summarise the weeks inside the eligibility window, separately for the Hot 100 (`h_`, songs) and the
Billboard 200 (`b_`, albums):

* `best_peak` — best position reached (missing if the artist did not chart);
* `entries` — number of different songs / albums; `weeks` — chart-weeks summed over entries;
* `top10_weeks`, `no1_weeks` — chart-weeks in the top 10 and at No. 1;
* `share` — `weeks` divided by all chart slots in the window (100 or 200 per week), the artist's share of the chart;
* `new_in_window` — 1 if the artist's first appearance on either chart (since 1998) falls inside the window; relevant for Best New Artist.

In [4]:
charts = pd.concat([hot, b200], ignore_index=True)
charts = charts[charts.chart_week >= f"{FIRST_SEASON - 2}-01-01"]
perf_names = {p: split_credit(p) for p in charts.performer.unique()}

credits = charts[["chart", "chart_week", "title", "performer", "current_week"]].copy()
credits["artist_key"] = credits.performer.map(lambda p: [norm(n) for n in perf_names[p]])
credits = credits.explode("artist_key")
first_chart = credits.groupby("artist_key").chart_week.min()

display_name = {}
for names in perf_names.values():
    for n in names:
        display_name.setdefault(norm(n), n)

blocks = []
for t in range(FIRST_SEASON, FORECAST_SEASON + 1):
    lo, hi = window(t)
    w = credits[credits.chart_week.between(lo, hi)]
    for ch, pre, size in [("hot100", "h", 100), ("b200", "b", 200)]:
        c = w[w.chart == ch]
        n_weeks = c.chart_week.nunique()
        grp = c.groupby("artist_key")
        f = pd.DataFrame({
            f"{pre}_best_peak":   grp.current_week.min(),
            f"{pre}_entries":     grp.title.nunique(),
            f"{pre}_weeks":       grp.size(),
            f"{pre}_top10_weeks": c[c.current_week <= 10].groupby("artist_key").size(),
            f"{pre}_no1_weeks":   c[c.current_week == 1].groupby("artist_key").size(),
        })
        f[f"{pre}_share"] = f[f"{pre}_weeks"] / (size * n_weeks)
        blocks.append(f.assign(season=t).reset_index())

feat = pd.concat(blocks).groupby(["artist_key", "season"]).first().reset_index()
count_cols = [c for c in feat.columns if c.endswith(("_entries", "_weeks", "_share"))]
feat[count_cols] = feat[count_cols].fillna(0)
feat["h_charted"] = feat.h_best_peak.notna().astype(int)
feat["b_charted"] = feat.b_best_peak.notna().astype(int)
feat["new_in_window"] = (feat.artist_key.map(first_chart) >= feat.season.map(lambda t: window(t)[0])).astype(int)
print(feat.groupby("season").size().rename("artists in pool").describe().round(0))

count      27.0
mean      944.0
std       222.0
min       631.0
25%       764.0
50%       879.0
75%      1120.0
max      1384.0
Name: artists in pool, dtype: float64


## 5. Who was nominated: performers of each nomination

The Grammy file lists the credited producers, engineers or video directors instead of the performer for most non-winning nominees
(and only the winner's name in several pop categories). We resolve the performer of each nomination in this order:

1. the Wikipedia table of the category (matched on season, category and title);
2. the performer in parentheses after the songwriters (Song of the Year);
3. the most-charted Billboard performer of the same title around the season;
4. the name field itself when it is a performer (winners) — for Best New Artist the nominee *is* the artist.

Season 2025 (68th Grammys) comes from its own file.

In [5]:
g = grammy[grammy.category.isin(RAW_TO_CAT) & grammy.year.between(FIRST_SEASON, 2024)].copy()
g["cat"] = g.category.map(RAW_TO_CAT)
g = g.rename(columns={"year": "season", "song_or_album": "title", "winner": "won"})

wiki["season"] = wiki.ceremony_year - 1
wiki["tkey"] = wiki.title.map(norm)
wiki_lookup = wiki.groupby(["season", "cat", "tkey"]).performer.first()

titles = credits.assign(tkey=credits.title.map(norm))
def billboard_performer(title, t, cat):
    lo, hi = window(t)
    src = titles[(titles.chart == ("b200" if cat in ALBUM_CATS else "hot100")) &
                 titles.chart_week.between(lo - pd.DateOffset(years=1), hi + pd.DateOffset(months=6))]
    m = src[src.tkey == norm(title)]
    return None if m.empty else m.groupby("performer").size().idxmax()

def wiki_performer(row):
    key = (row.season, row["cat"], norm(row.title))
    if key in wiki_lookup.index:
        return wiki_lookup[key]
    # titles written differently, e.g. "F**k You" / "Fuck You": accept a unique containment match
    sub = wiki[(wiki.season == row.season) & (wiki["cat"] == row["cat"])]
    tk = norm(row.title)
    hit = sub[sub.tkey.map(lambda w: len(w) > 3 and len(tk) > 3 and (w in tk or tk in w))]
    return hit.performer.iloc[0] if len(hit) == 1 else None

def resolve(row):
    if row["cat"] == "best_new_artist":
        return row.title, "nominee name"
    p = wiki_performer(row)
    if p: return p, "wikipedia"
    art = row.artist if isinstance(row.artist, str) else ""
    m = re.search(r"\(([^()]*)\)\s*$", art)
    if m and "songwriter" in art.lower(): return m.group(1), "songwriters (performer)"
    p = billboard_performer(row.title, row.season, row["cat"])
    if p: return p, "billboard title"
    if art and not re.search(r"producer|engineer|director|mixer|songwriter|artist", art, re.I):
        return art.strip('"'), "name field"
    return None, "unresolved"

res = g.apply(resolve, axis=1, result_type="expand")
g["performer"], g["resolved_by"] = res[0], res[1]

g68_ = g68.assign(season=2025, won=g68.winner.astype(bool), resolved_by="68th list")
noms = pd.concat([g[["season", "cat", "title", "performer", "won", "resolved_by"]],
                  g68_[["season", "cat", "title", "performer", "won", "resolved_by"]]], ignore_index=True)
print(noms.resolved_by.value_counts().to_string())

resolved_by
wikipedia                  1043
nominee name                150
68th list                    52
billboard title              25
unresolved                    4
songwriters (performer)       4


In [6]:
# nominations still without a performer
noms[noms.resolved_by == "unresolved"][["season", "cat", "title"]]

,season,cat,title
8,2024,album_of_the_year,Djesse Vol. 4
10,2024,album_of_the_year,Chappell Roan The Rise And Fall Of A Midwest P...
327,2019,album_of_the_year,Norman F***ing Rockwell!
660,2011,pop_solo,F***in’ Perfect


### Check: number of nominations per category and season

Most categories have 5 nominees per season; the general field had 8–10 since season 2018, and the solo category had about 10 before 2012
(female + male). A season with fewer rows than expected points to a gap in the Grammy file.

In [7]:
counts = noms.pivot_table(index="season", columns="cat", values="title", aggfunc="size").fillna(0).astype(int)
counts

cat,album_of_the_year,best_new_artist,music_video,pop_duo_group,pop_solo,pop_vocal_album,record_of_the_year,song_of_the_year
season,,,,,,,,
2000,5,5,5,10,11,5,5,5
2001,5,5,5,10,10,5,5,5
2002,5,5,5,11,10,5,5,5
2003,5,5,5,10,10,5,5,5
2004,5,5,5,10,10,5,5,5
2005,5,5,5,10,10,5,5,5
2006,5,5,5,10,10,5,5,5
2007,5,5,5,10,10,5,5,5
2008,5,5,5,10,12,5,5,5


## 6. Labels and previous Grammy record

Each nomination is credited to every artist in its performer credit (both "Lady Gaga" and "Bruno Mars" for "Die with a Smile").
`nominated = 1` if the artist has at least one nomination in the category and season.

Previous Grammy record uses **earlier seasons only** (no information from the season being forecast):
`prior_noms` and `prior_wins` across the eight categories, and `prior_noms_same_cat` in the same category.

In [8]:
noms["artists"] = noms.performer.map(lambda p: split_credit(p) if isinstance(p, str) else [])
lab = noms.explode("artists").dropna(subset=["artists"])
lab["artist_key"] = lab.artists.map(norm)
for k, a in zip(lab.artist_key, lab.artists):
    display_name.setdefault(k, a)
labels = (lab.groupby(["artist_key", "season", "cat"]).won.max().astype(int)
          .reset_index().assign(nominated=1))

prior = []
for t in range(FIRST_SEASON, FORECAST_SEASON + 1):
    past = labels[labels.season < t]
    p = past.groupby("artist_key").agg(prior_noms=("nominated", "sum"), prior_wins=("won", "sum"))
    pc = past.groupby(["artist_key", "cat"]).size().rename("prior_noms_same_cat").reset_index()
    prior.append((p.reset_index().assign(season=t), pc.assign(season=t)))
prior_all = pd.concat([p for p, _ in prior])
prior_cat = pd.concat([pc for _, pc in prior])

In [9]:
# check: nominations credited to one artist in one season
lab[(lab.artist_key == norm("Sabrina Carpenter")) & (lab.season == 2024)][["season", "cat", "title", "performer", "won"]]

,season,cat,title,performer,won
4,2024,record_of_the_year,Espresso,Sabrina Carpenter,False
6,2024,album_of_the_year,Short n' Sweet,Sabrina Carpenter,False
12,2024,best_new_artist,Sabrina Carpenter,Sabrina Carpenter,False
21,2024,pop_vocal_album,Short n' Sweet,Sabrina Carpenter,True
31,2024,pop_solo,Espresso,Sabrina Carpenter,True
48,2024,song_of_the_year,Please Please Please,Sabrina Carpenter,False


## 7. Assemble artist × season × category

In [10]:
pool = feat[(feat.h_charted == 1) | (feat.b_charted == 1)]
data = pool.merge(pd.DataFrame({"cat": CATS}), how="cross")
data = data.merge(labels, on=["artist_key", "season", "cat"], how="left")
data[["nominated", "won"]] = data[["nominated", "won"]].fillna(0)
data.loc[data.season > LAST_LABELLED, ["nominated", "won"]] = np.nan     # season 2026 = forecast target
data = (data.merge(prior_all, on=["artist_key", "season"], how="left")
            .merge(prior_cat, on=["artist_key", "season", "cat"], how="left"))
data[["prior_noms", "prior_wins", "prior_noms_same_cat"]] = \
    data[["prior_noms", "prior_wins", "prior_noms_same_cat"]].fillna(0).astype(int)
data["artist"] = data.artist_key.map(display_name)

COLS = ["season", "cat", "artist", "artist_key", "nominated", "won",
        "h_best_peak", "h_entries", "h_weeks", "h_top10_weeks", "h_no1_weeks", "h_share", "h_charted",
        "b_best_peak", "b_entries", "b_weeks", "b_top10_weeks", "b_no1_weeks", "b_share", "b_charted",
        "new_in_window", "prior_noms", "prior_wins", "prior_noms_same_cat"]
data = data[COLS].sort_values(["season", "cat", "artist_key"]).reset_index(drop=True)
print(data.shape)
data[(data.artist == "Sabrina Carpenter") & data.season.between(2023, 2026) & (data.cat == "pop_vocal_album")]

(203944, 24)


,season,cat,artist,artist_key,nominated,won,h_best_peak,h_entries,h_weeks,h_top10_weeks,...,b_entries,b_weeks,b_top10_weeks,b_no1_weeks,b_share,b_charted,new_in_window,prior_noms,prior_wins,prior_noms_same_cat
186830,2023,pop_vocal_album,Sabrina Carpenter,sabrinacarpenter,0.0,0.0,56.0,1.0,19.0,0.0,...,1.0,6.0,0.0,0.0,0.000566,1,0,0,0,0
192294,2024,pop_vocal_album,Sabrina Carpenter,sabrinacarpenter,1.0,1.0,1.0,13.0,111.0,41.0,...,4.0,50.0,4.0,3.0,0.004808,1,0,0,0,0
197456,2025,pop_vocal_album,Sabrina Carpenter,sabrinacarpenter,1.0,0.0,1.0,25.0,222.0,35.0,...,4.0,97.0,47.0,2.0,0.009327,1,0,6,2,1
202555,2026,pop_vocal_album,Sabrina Carpenter,sabrinacarpenter,NaN,NaN,6.0,12.0,99.0,3.0,...,3.0,108.0,14.0,0.0,0.010385,1,0,12,2,2


## 8. Pool coverage

A nominee who never charted in the window is not in the pool, so the model cannot give it a probability. **Coverage** is the share of
nominated artists who are in the pool; nominations with an unknown performer count as misses. Low coverage means that a top-5 list
drawn from the pool would miss real nominees, whatever the model.

In [11]:
in_pool = set(zip(pool.artist_key, pool.season))
lab_l = labels[labels.season <= LAST_LABELLED].copy()
lab_l["in_pool"] = [(k, s) in in_pool for k, s in zip(lab_l.artist_key, lab_l.season)]
unres = noms[noms.resolved_by == "unresolved"].groupby("cat").size()
cov = lab_l.groupby("cat").agg(nominated_artists=("in_pool", "size"), in_pool=("in_pool", "sum"))
cov["unresolved_nominations"] = unres.reindex(cov.index).fillna(0).astype(int)
cov["coverage"] = cov.in_pool / (cov.nominated_artists + cov.unresolved_nominations)
cov_recent = lab_l[lab_l.season >= 2012].groupby("cat").in_pool.mean().rename("coverage_since_2012")
cov.join(cov_recent).round(3)

,nominated_artists,in_pool,unresolved_nominations,coverage,coverage_since_2012
cat,,,,,
album_of_the_year,159,152,3,0.938,0.969
best_new_artist,159,142,0,0.893,0.838
music_video,164,129,0,0.787,0.791
pop_duo_group,313,268,0,0.856,0.933
pop_solo,189,177,1,0.932,0.972
pop_vocal_album,135,127,0,0.941,0.945
record_of_the_year,199,189,0,0.950,0.937
song_of_the_year,190,184,0,0.968,0.968


In [12]:
# examples of nominated artists outside the pool (did not chart in the window)
outside = lab_l[~lab_l.in_pool].assign(artist=lambda d: d.artist_key.map(display_name))
outside[outside.season >= 2018].sort_values(["season", "cat"])[["season", "cat", "artist", "won"]].head(25)

,season,cat,artist,won
1429,2018,music_video,Tierra Whack,0
229,2018,pop_duo_group,Bradley Cooper,1
1191,2018,pop_vocal_album,Pink,0
230,2018,record_of_the_year,Bradley Cooper,0
231,2018,song_of_the_year,Bradley Cooper,0
202,2019,best_new_artist,Black Pumas,0
1257,2019,best_new_artist,Rosalia,0
1380,2019,best_new_artist,Tank And The Bangas,0
1502,2019,best_new_artist,Yola,0
322,2019,music_video,The Chemical Brothers,0


## 9. Other music awards: AMA, VMA, iHeartRadio

Each award file has one row per artist and ceremony year with 0/1 flags for nominations and wins by category.
They are added to the table above with a **left join on the key `(artist_key, season)`**: every row is kept, and an artist without
nominations at an award gets zeros. Award features describe the artist in the season, so they are the same in the artist's eight category rows.

**From ceremony year to Grammy season, without look-ahead.** A feature may be used for season `t` only if it was public before the
Grammy nominations of `t` (November of `t`).

| Award | Ceremony | Season | Nominations | Winners |
|---|---|---|---|---|
| American Music Awards | January of `Y` (≤ 2002), music of `Y−1` | `Y−1` | same season | previous ceremony only |
| | November of `Y` (2003–2022) | `Y` | same season (announced in October) | previous ceremony only (often after the Grammy nominations) |
| | none in 2023–2024; May of `Y` (2025–2026) | `Y` | same season | previous ceremony only |
| MTV Video Music Awards | late August / September of `Y` | `Y` | same season | same season (before November) |
| iHeartRadio Music Awards | March–May of `Y`, music of `Y−1` | `Y` | same season | same season (before November) |

iHeartRadio honours the music of the previous year, but its nominations are announced in January of `Y`, after the Grammy nominations of
season `Y−1`, so they can only be used for season `Y`. For uniformity AMA winners are always lagged, also for the May ceremonies.
The AMAs held two ceremonies in 2003 (January and November) that the file labels both as 2003; all are assigned to season 2003,
so season 2002 has no AMA data. A flag `<award>_held` marks seasons with a ceremony, so "no nomination" and "no ceremony that year" differ.

In [13]:
AWARD_SEASON = {
    "ama":    lambda y: y - 1 if y <= 2002 else y,
    "vma":    lambda y: y,
    "iheart": lambda y: y,
}
LAG_WINNERS = {"ama": True, "vma": False, "iheart": False}

# names written differently on Billboard, and duos credited jointly by an award but split on Billboard
ALIASES = {
    "pink": ["pnk", "pink"], "kesha": ["keha", "kesha"], "tydollasign": ["tydollaign"],
    "ladya": ["ladya", "ladyantebellum"], "macklemoreryanlewis": ["macklemore", "ryanlewis"],
    "silksonic": ["silksonic", "brunomars", "andersonpaak"],
}
pool_keys = set(zip(data.artist_key, data.season))

def to_keys(name, season):
    k = norm(name)
    cands = ALIASES.get(k, [k])
    hits = [c for c in cands if (c, season) in pool_keys]
    return hits if hits else [k]

def award_table(award, raw):
    d = raw.copy()
    d["season"] = d.year.map(AWARD_SEASON[award])
    nom = [c for c in d.columns if c.endswith("_nominated")]
    win = [c for c in d.columns if c.endswith("_winner")]
    d[nom + win] = d[nom + win].fillna(0).astype(int)
    d["artist_key"] = [to_keys(a, s) for a, s in zip(d.artist, d.season)]
    d = d.explode("artist_key").groupby(["artist_key", "season"])[nom + win].max().reset_index()
    d[f"{award}_noms_total"] = d[nom].sum(axis=1)
    d[f"{award}_wins_total"] = d[win].sum(axis=1)
    return d, nom, win

award_tables, match_report = {}, []
for award, raw in awards_raw.items():
    d, nom, win = award_table(award, raw)
    award_tables[award] = (d, nom, win)
    recent = d[d.season >= FIRST_SEASON]
    hit = np.array([(k, s) in pool_keys for k, s in zip(recent.artist_key, recent.season)])
    match_report.append({"award": award, "artist-seasons since 2000": len(recent), "in pool": hit.sum(),
                         "share": round(hit.mean(), 3),
                         "not in pool (examples)": ", ".join(recent.artist_key[~hit].head(8))})
pd.DataFrame(match_report)

,award,artist-seasons since 2000,in pool,share,not in pool (examples)
0,ama,474,460,0.970,"audreynuna, bjork, carolinagaitan, ejae, huntr..."
1,vma,797,767,0.962,"alltimelow, angelsairwaves, ashanti, cnco, cnc..."
2,iheart,454,418,0.921,"anitta, atljacob, audreynuna, beberexha, bella..."


Artists that are not found did not chart in that season's window (for example the cast of *Encanto*, or the members of the fictional
group Huntr/x credited separately), so they are outside the candidate pool and their rows are not needed.

In [14]:
full = data.copy()
seasons = sorted(full.season.unique())
for award, (d, nom, win) in award_tables.items():
    held = sorted(d.season.unique())
    full[f"{award}_held"] = full.season.isin(held).astype(int)
    same = nom + [f"{award}_noms_total"] + ([] if LAG_WINNERS[award] else win + [f"{award}_wins_total"])
    full = full.merge(d[["artist_key", "season"] + same], on=["artist_key", "season"], how="left")
    if LAG_WINNERS[award]:
        # winners of the most recent ceremony of an EARLIER season
        prev = {s: max([h for h in held if h < s], default=np.nan) for s in seasons}
        lagged = d[["artist_key", "season"] + win + [f"{award}_wins_total"]].rename(
            columns={**{c: c + "_prev" for c in win}, f"{award}_wins_total": f"{award}_wins_prev_total", "season": "prev_season"})
        full["prev_season"] = full.season.map(prev)
        full = full.merge(lagged, on=["artist_key", "prev_season"], how="left").drop(columns="prev_season")

award_cols = [c for c in full.columns if c not in data.columns]
full[award_cols] = full[award_cols].fillna(0).astype(int)
assert len(full) == len(data), "left join must keep every row"
assert not full.duplicated(["season", "cat", "artist_key"]).any()
print(full.shape, f"— {len(award_cols)} award columns added")

(203944, 95) — 71 award columns added


## 10. Checks and output

In [15]:
# artists with a nomination at each award, per season (0 only where no ceremony)
one = full.drop_duplicates(["artist_key", "season"])
pd.DataFrame({a: one.groupby("season")[f"{a}_noms_total"].apply(lambda x: (x > 0).sum()) for a in award_tables}).T

season,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
ama,11,10,0,25,15,13,18,14,15,12,...,26,20,22,22,28,30,0,0,27,33
vma,18,28,24,25,25,27,28,34,21,19,...,34,29,28,35,34,36,40,38,41,33
iheart,0,0,0,0,0,0,0,0,0,0,...,33,42,37,29,34,26,44,47,28,37


In [16]:
# Grammy nomination rate in each category: artists with vs without a nomination at each award (labelled seasons)
lab = full[full.season <= LAST_LABELLED]
rows = {}
for a in award_tables:
    held = lab[lab[f"{a}_held"] == 1]
    rows[(a, "nominee")] = held[held[f"{a}_noms_total"] > 0].groupby("cat").nominated.mean()
    rows[(a, "not nominee")] = held[held[f"{a}_noms_total"] == 0].groupby("cat").nominated.mean()
pd.DataFrame(rows).round(3)

ama                 vma              iheart            
                   nominee not nominee nominee not nominee nominee not nominee
cat                                                                           
album_of_the_year    0.145       0.003   0.112       0.003   0.050       0.006
best_new_artist      0.082       0.004   0.082       0.003   0.016       0.006
music_video          0.077       0.004   0.072       0.003   0.031       0.004
pop_duo_group        0.225       0.006   0.159       0.006   0.084       0.007
pop_solo             0.155       0.004   0.114       0.004   0.052       0.004
pop_vocal_album      0.133       0.003   0.094       0.002   0.060       0.004
record_of_the_year   0.222       0.003   0.153       0.003   0.071       0.007
song_of_the_year     0.187       0.004   0.135       0.004   0.073       0.008

In [17]:
(full[(full.artist_key == "sabrinacarpenter") & (full.cat == "pop_vocal_album") & full.season.between(2022, 2026)]
 [["season", "nominated", "h_best_peak", "prior_noms", "ama_noms_total", "vma_noms_total", "vma_wins_total", "iheart_noms_total"]])

,season,nominated,h_best_peak,prior_noms,ama_noms_total,vma_noms_total,vma_wins_total,iheart_noms_total
181154,2022,0.0,NaN,0,0,0,0,0
186830,2023,0.0,56.0,0,0,0,0,0
192294,2024,1.0,1.0,0,0,3,1,0
197456,2025,1.0,1.0,6,6,2,0,5
202555,2026,NaN,6.0,12,7,5,1,5


In [18]:
full.to_csv("grammy_nomination_dataset.csv", index=False)
print("saved grammy_nomination_dataset.csv", full.shape)

saved grammy_nomination_dataset.csv (203944, 95)
